# CardioIA — Fase 6 | Modelo preditivo de pico de risco
**FIAP • Etapa 1 — Machine Learning supervisionado**

Notebook reprodutível baseado na base sintética e nas regras fornecidas no enunciado. **Uso exclusivamente acadêmico, sem validade clínica.**

## 1. Bibliotecas e reprodutibilidade
Utilizamos bibliotecas comuns de ciência de dados. A semente 42 reproduz a base disponibilizada pela disciplina.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay)
SEED = 42
OUT = Path('artefatos')
OUT.mkdir(exist_ok=True)
print('Ambiente preparado.')

## 2. Geração da base sintética
As distribuições e limiares abaixo foram preservados do enunciado. O alvo é 1 quando **pelo menos duas** das seis condições de risco são verdadeiras.

In [ ]:
np.random.seed(42)
n = 5000
df = pd.DataFrame({
    "idade_media": np.random.normal(55, 12, n).clip(18, 90),
    "carga_sistema": np.random.uniform(30, 100, n),
    "tempo_espera": np.random.normal(45, 20, n).clip(5, 180),
    "fc_media": np.random.normal(85, 15, n).clip(50, 150),
    "spo2_media": np.random.normal(95, 3, n).clip(70, 100),
    "pa_sistolica": np.random.normal(125, 20, n).clip(80, 200),
    "temperatura": np.random.normal(24, 5, n).clip(10, 40),
    "densidade_eventos": np.random.poisson(3, n),
    "eventos_24h": np.random.poisson(2, n),
    "disponibilidade_recursos": np.random.uniform(20, 100, n)
})
risk_score = (
    (df['carga_sistema'] > 80).astype(int)
    + (df['tempo_espera'] > 60).astype(int)
    + (df['spo2_media'] < 92).astype(int)
    + (df['fc_media'] > 110).astype(int)
    + (df['eventos_24h'] > 4).astype(int)
    + (df['disponibilidade_recursos'] < 40).astype(int)
)
df['pico_risco'] = (risk_score >= 2).astype(int)
df.to_csv(OUT / 'base_sintetica.csv', index=False)
print('Formato:', df.shape)
print('Distribuição da classe (%):')
print((df['pico_risco'].value_counts(normalize=True).sort_index()*100).round(2))
display(df.head())

## 3. Qualidade e exploração dos dados
Verificamos dados ausentes, duplicados e distribuição do alvo. Duplicatas não são removidas automaticamente, pois podem representar coincidências legítimas na simulação.

In [ ]:
print('Ausentes:', int(df.isna().sum().sum()))
print('Duplicados:', int(df.duplicated().sum()))
display(df.describe().T.round(2))
fig, ax = plt.subplots(figsize=(6, 4))
df['pico_risco'].value_counts().sort_index().plot.bar(ax=ax)
ax.set(title='Distribuição do alvo', xlabel='Pico de risco (0 = não; 1 = sim)', ylabel='Registros')
plt.tight_layout()
plt.savefig(OUT / 'distribuicao_alvo.png', dpi=160)
plt.show()

## 4. Separação treino/teste
Separamos 80% para treinamento e validação cruzada e 20% para teste final, com estratificação. O conjunto de teste não participa da seleção do algoritmo.

In [ ]:
FEATURES = [c for c in df.columns if c != 'pico_risco']
X, y = df[FEATURES], df['pico_risco']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)
print('Treino:', X_train.shape, '| Teste:', X_test.shape)

## 5. Treinamento e comparação
A Regressão Logística utiliza padronização **dentro do Pipeline** para evitar vazamento de informações entre os folds. A Random Forest é avaliada com os mesmos folds estratificados. A seleção usa F1-score médio de validação cruzada.

In [ ]:
models = {
    'Regressão Logística': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=2000, random_state=SEED))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, min_samples_leaf=2, random_state=SEED, n_jobs=-1)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, scoring='f1', cv=cv)
    results.append({'modelo': name, 'f1_cv_medio': scores.mean(), 'f1_cv_desvio': scores.std()})
comparison = pd.DataFrame(results).sort_values('f1_cv_medio', ascending=False)
display(comparison.round(4))
comparison.to_csv(OUT / 'comparacao_modelos.csv', index=False)
best_name = comparison.iloc[0]['modelo']
best_model = models[best_name].fit(X_train, y_train)
print('Modelo escolhido:', best_name)

## 6. Avaliação final
Acurácia, precisão, recall, F1, ROC-AUC e matriz de confusão são calculados **uma única vez no conjunto de teste**. Em uma aplicação real, falsos negativos exigiriam atenção especial; aqui os rótulos são sintéticos.

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
metrics = {
    'modelo': best_name,
    'acuracia': float(accuracy_score(y_test, y_pred)),
    'precisao': float(precision_score(y_test, y_pred, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred, zero_division=0)),
    'f1': float(f1_score(y_test, y_pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, y_prob)),
    'matriz_confusao': confusion_matrix(y_test, y_pred).tolist()
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print('Relatório por classe:', classification_report(y_test, y_pred, zero_division=0))
with open(OUT / 'metricas.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Sem pico', 'Com pico'], ax=ax, cmap='Blues')
plt.tight_layout()
plt.savefig(OUT / 'matriz_confusao.png', dpi=160)
plt.show()
fig, ax = plt.subplots(figsize=(6, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
plt.tight_layout()
plt.savefig(OUT / 'curva_roc.png', dpi=160)
plt.show()

## 7. Persistência do modelo
O arquivo `.joblib` será consumido pela tool do Agente Analista de Risco. **Nunca carregue arquivos joblib de fontes desconhecidas.**

In [ ]:
joblib.dump(best_model, OUT / 'modelo_risco.joblib')
with open(OUT / 'colunas_modelo.json', 'w', encoding='utf-8') as f:
    json.dump(FEATURES, f, ensure_ascii=False, indent=2)
print('Modelo e esquema salvos em', OUT.resolve())

## 8. Simulação de um novo caso
O enunciado usa `disponibilidade recursos` com espaço no exemplo. Corrigimos para `disponibilidade_recursos`, nome presente na base de treinamento. A probabilidade é uma estimativa do classificador para **pico_risco sintético**, não uma probabilidade clínica calibrada.

In [ ]:
novo_caso = pd.DataFrame([{
    'idade_media': 62,
    'carga_sistema': 92,
    'tempo_espera': 75,
    'fc_media': 118,
    'spo2_media': 89,
    'pa_sistolica': 100,
    'temperatura': 37.5,
    'densidade_eventos': 7,
    'eventos_24h': 5,
    'disponibilidade_recursos': 25
}])[FEATURES]
modelo_carregado = joblib.load(OUT / 'modelo_risco.joblib')
prob = float(modelo_carregado.predict_proba(novo_caso)[0, 1])
classe = int(modelo_carregado.predict(novo_caso)[0])
print(f'Probabilidade estimada de pico_risco: {prob:.4f}')
print('Classificação:', 'PICO DE RISCO' if classe else 'SEM PICO DE RISCO')
assert novo_caso.columns.tolist() == FEATURES
assert 0 <= prob <= 1

## 9. Protocolos simulados aplicáveis
Os protocolos abaixo são **regras fictícias do enunciado**, não recomendações médicas. São filtrados por limiares estritos (`>` ou `<`). O segundo agente realizará essa consulta na próxima etapa.

In [ ]:
protocolos = [
    ('spo2_media', '<', 88, 'Iniciar suporte respiratório imediato'),
    ('carga_sistema', '>', 85, 'Abrir unidades temporárias'),
    ('fc_media', '>', 120, 'Iniciar monitoramento intensivo'),
    ('eventos_24h', '>', 5, 'Ativar equipe de resposta rápida'),
    ('disponibilidade_recursos', '<', 30, 'Redistribuir insumos'),
    ('tempo_espera', '>', 90, 'Priorizar triagem emergencial'),
    ('temperatura', '>', 38, 'Iniciar protocolo infeccioso'),
    ('densidade_eventos', '>', 6, 'Ativar plano de contingência'),
    ('pa_sistolica', '<', 90, 'Iniciar protocolo de choque')
]
registro = novo_caso.iloc[0]
aplicaveis = [f'{campo} {operador} {limiar}: {acao}'
              for campo, operador, limiar, acao in protocolos
              if (registro[campo] < limiar if operador == '<' else registro[campo] > limiar)]
# A regra 'risco sistêmico alto' não define limiar no enunciado;
# sua ativação depende de uma definição explícita a ser documentada.
print('Protocolos aplicáveis:')
for item in aplicaveis:
    print('-', item)
with open(OUT / 'exemplo_saida.json', 'w', encoding='utf-8') as f:
    json.dump({'probabilidade': prob, 'classe': classe, 'protocolos': aplicaveis},
              f, ensure_ascii=False, indent=2)

## 10. Limitações e próximos passos
- A variável alvo é construída por regras fixas, não por desfechos clínicos observados.
- O conjunto é sintético; métricas elevadas não indicam capacidade de generalização clínica.
- A saída `predict_proba` não deve ser tratada como probabilidade clínica calibrada.
- Variáveis misturam características de pacientes e do sistema de atendimento.
- O protocolo “risco sistêmico alto” não possui definição quantitativa no enunciado e precisa de critério explícito.
- Próxima etapa: integrar o artefato salvo ao OpenAI Agents SDK, com handoffs, tools, histórico e validação de saída.